# 04 — Feature Engineering & Spark Scaling


In [ ]:
import sys
from pathlib import Path
REPO = Path.cwd()
for candidate in [REPO, REPO.parent, REPO.parent.parent]:
    if (candidate / "src" / "neuro").exists():
        REPO = candidate
        break
sys.path.insert(0, str(REPO / "src"))
import os
os.chdir(REPO)
%matplotlib inline


In [ ]:

from pyspark.sql import functions as F
from pyspark.sql.types import FloatType, StructType, StructField, StringType, IntegerType
from neuro.bids import validate_bids
from neuro.features import build_feature_table, save_features_parquet, roi_summary_features, extract_roi_timeseries, get_schaefer_masker
from neuro.spark_utils import get_spark

report = validate_bids()
runs = report["runs"]
spark = get_spark("BladerunnerNeuro_Features")
runs_pdf = runs[runs["bold_exists"]].copy()
print(f"Building features for {len(runs_pdf)} runs")


In [ ]:

# Spark UDF wrapper for ROI mean (distributed metadata aggregation)
masker = get_schaefer_masker(n_rois=50)

@F.udf(returnType=FloatType())
def roi_mean_scalar(path):
    ts = extract_roi_timeseries(path, masker)
    return float(ts.mean())

files_sdf = spark.createDataFrame(runs_pdf[["subject","task","run","bold_path","group_short"]])
files_sdf = files_sdf.withColumn("roi_global_mean", roi_mean_scalar(F.col("bold_path")))
files_sdf.groupBy("group_short").avg("roi_global_mean").show()


In [ ]:

# Full feature table (local; saves parquet for TF)
feat_df = build_feature_table(runs_pdf, n_rois=50)
path = save_features_parquet(feat_df)
print("Saved:", path)
feat_df[["subject","task","ts_mean","ts_entropy"]].head()


In [ ]:

from pathlib import Path
nb_name = Path.cwd().name if False else "04_feature_engineering"
!jupyter nbconvert --to html notebooks/04_feature_engineering.ipynb --output-dir notebooks/html 2>/dev/null || true
